# Business Actions 

The model was designed to identify at-risk enterprise clients three months prior to renewal, enabling proactive Customer Success interventions. High-risk accounts were further segmented by engagement, adoption, and satisfaction signals to support targeted remediation strategies. Scenario analysis suggests that focusing on the highest-risk segment could prevent a substantial fraction of non-renewals and protect significant recurring revenue.

The Goal is to translate model outputs into: 
- Who should Customer Sucess act on
- What actions to take 
- Why those actions make sense 
- Expected business impact 

Tailored intervention recommendation considering common limitations in an Enterprise B2B based on risk drivers. Common limitations:
- Customer Success cannot intervene on every account 
- Need to prioritize high-risk, high-value clients 

In [0]:
from pyspark.sql import functions as F

In [0]:
ranked_preds = spark.table("renewal_model_ranked_predictions")
display(
    ranked_preds.select(
        "client_id",
        "renewal_cycle_id",
        "industry",
        "company_size",
        "non_renewal_probability",
        "risk_bucket",
        "non_renewal_flag"
    )
)

In [0]:
# Focus on High-Risk accounts, meaning accounts that would require proactive intervention 3 months before renewal
high_risk_accounts = ranked_preds.filter(F.col("risk_bucket") == "High")

display(
    high_risk_accounts
    .orderBy(F.desc("non_renewal_probability"))
)

In [0]:
# Quantify expected churn in high-risk segment
display(
    high_risk_accounts.agg(
        F.count("*").alias("accounts"),
        F.sum("non_renewal_flag").alias("actual_churners"),
        F.avg("non_renewal_flag").alias("churn_rate"),
        F.avg("non_renewal_probability").alias("avg_predicted_risk")
    )
)

- 10 accounts 
- 5 actual churners 
- Churn Rate of 50%

In [0]:
# Identify key risk drivers for these accounts 
display(
    high_risk_accounts.agg(
        F.avg("utilization_change_3m_vs_prior_3m").alias("avg_utilization_change"),
        F.avg("active_members_change_3m_vs_prior_3m").alias("avg_active_members_change"),
        F.avg("nps_change_3m_vs_prior_3m").alias("avg_nps_change"),
        F.avg("webinar_attendance_change_3m_vs_prior_3m").alias("avg_webinar_change")
    )
)

- **Negative Values**: declining engagement 
- **Positive Values**: improving engagement

In [0]:
# Segment high-risk accounts by likely root cause because it would not be a generic intervention for all. There would be tailor actions to the problem 
engagement_risk = high_risk_accounts.filter(
    F.col("utilization_change_3m_vs_prior_3m") < -0.2
)

satisfaction_risk = high_risk_accounts.filter(
  F.col("nps_change_3m_vs_prior_3m") < -10
)

adoption_risk = high_risk_accounts.filter(
  F.col("active_members_change_3m_vs_prior_3m") < -0.15
)

**High-risk engagement decline actions**
- Executive check-in with employer
- Custom engagement plan
- Targeted communication campaign 
- Onboarding refresh for new employees

**Satisfaction decline actions**
- Issue resolution review 
- Product feedback sessions 
- Service improvements 
- Dedicated support 

**Adoption decline actions**
- Training sessions 
- Feature education 
- Champion re-engagement 
- Usage incentives 


In [0]:
# Simulate operational prioritization assuming that Customer Success can only handle 30 accounts
top_30_accounts = (
    ranked_preds
    .orderBy(F.desc("non_renewal_probability"))
    .limit(30)
)

display(top_30_accounts)

In [0]:
# Estimate churn captured 
display(
    top_30_accounts.agg(
        F.count("*").alias("accounts"),
        F.sum("non_renewal_flag").alias("actual_churners"),
        F.avg("non_renewal_flag").alias("churn_rate")
    )
)

In [0]:
# Estimate business impact 
avg_contract_value = 250000 # Assumption
success_rate = 0.3 # Assumption

estimated_churners = (
    top_30_accounts
    .filter(F.col("non_renewal_flag") == 1)
    .count()
)

estimated_saved = estimated_churners * success_rate
estimated_revenue_saved = estimated_saved * avg_contract_value

print("Estimated churners in top 30:", estimated_churners)
print("Estimated contracts saved:", estimated_saved)
print("Estimated revenue protected:", estimated_revenue_saved)

In [0]:
# Create actionable output table 
action_table = ranked_preds.select(
    "client_id",
    "renewal_cycle_id",
    "industry",
    "company_size",
    "non_renewal_probability",
    "risk_bucket"
)

display(action_table.orderBy(F.desc("non_renewal_probability")))